# 13. Сохранение и загрузка модели

Видео: [13. Сохранение и загрузка](https://www.youtube.com/watch?v=gVyK6Riky20&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=14)

## О чём урок

После обучения веса живут только в RAM. Нужно:
1. **сохранить** лучшее состояние на диск;
2. **загрузить** в новую модель той же архитектуры;
3. (опционально) сохранить ещё optimizer/scheduler/эпоху — чтобы **продолжить** обучение после сбоя.

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11.5**. Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

| способ | что в файле | когда |
|:---|:---|:---|
| `torch.save(model, …)` | вся модель целиком | **нежелательно** (хрупко между версиями/ОС) |
| `torch.save(model.state_dict(), …)` | только веса | **рекомендуется** |
| `torch.save({...}, …)` | веса + opt + sched + история | продолжить обучение |

**Когда сохранять:** если **val** loss стал лучше прежнего (не train).

**Как смотреть.** ~16 мин — целиком или кусками: save/load → best в цикле → checkpoint → `map_location`.

**Как решать.** Замени `...` → проверка. Решения — `13_save_load_решения.ipynb` после попытки.

In [75]:
import copy
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
device = torch.device("cpu")

CKPT = Path("_checkpoints_13")
CKPT.mkdir(exist_ok=True)


def need(cond, msg):
    assert cond, msg


class TinyReg(nn.Module):
    """y ≈ a·x + b — крошечная регрессия для save/load."""

    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(1, 1)

    def forward(self, x):
        return self.lin(x)


# --- учебные данные (НЕ главная тема урока) ---
# Нужны только чтобы был цикл train→val→save. Это не Fashion и не «реальная» задача.
# X — ровная сетка признаков от -1 до 1, форма (80, 1) для Linear(1, 1).
# y — ответы по выдуманной формуле y ≈ 2x+1; шум (+0.05*randn) просто «как в жизни»,
#     на понимание save/load не влияет — можно было и без шума.
# Формулу 2x+1 тоже просто придумали для бутафории.
X = torch.linspace(-1, 1, 80).unsqueeze(1)
y = 2 * X + 1 + 0.05 * torch.randn_like(X)
ds = TensorDataset(X, y)
loader = DataLoader(ds, batch_size=16, shuffle=True)

print("torch", torch.__version__, "| CKPT:", CKPT.resolve())

torch 2.14.1+cpu | CKPT: C:\Users\HUAWEI\Desktop\учеба мл\pytorch\13\_checkpoints_13


## 1. <span style="color:#7CDBA0">torch.save / torch.load</span> — фундамент API

**Что делаем:** сохраняем на диск обычный Python-словарь с тензорами и читаем его обратно.

**Зачем:** понять сам API `save`/`load`. Тот же вызов потом используют для `state_dict` и checkpoint.

**Это НЕ модель.** `demo` — просто `dict` с ключами `"w"` / `"b"` (названия «как будто вес и bias»). Нейросети (`nn.Module`) здесь нет. Технически `torch.save` умеет сохранить и целую модель — так **нежелательно**; в следующем блоке сохраняем правильно: только `state_dict`.

```python
torch.save(obj, path)          # на диск
obj2 = torch.load(path, map_location="cpu", weights_only=True)  # с диска
```

Проверка: `loaded` совпадает с тем, что положили в `demo`.

In [76]:
# 1.1 save/load словаря тензоров (НЕ модель — просто dict для тренировки API)
path_d = CKPT / "demo_dict.pt"
demo = {
    "w": torch.tensor([1.0, 2.0]),
    "b": torch.tensor(0.5),
}  # учебные тензоры, не state_dict сети

torch.save(demo, path_d)
loaded = torch.load(path_d, map_location="cpu", weights_only=True)

print("loaded:", loaded)

loaded: {'w': tensor([1., 2.]), 'b': tensor(0.5000)}


In [77]:
need(path_d.is_file(), "сначала torch.save(demo, path_d)")
need(isinstance(loaded, dict) and "w" in loaded, "loaded = torch.load(...)")
need(torch.equal(loaded["w"], demo["w"]), "веса должны совпасть")
print("1.1 — верно")

1.1 — верно


## 2. <span style="color:#7CDBA0">state_dict</span> — сохраняем веса, не «всю модель»

```python
torch.save(model.state_dict(), "best.pt")

model2 = TinyReg()                       # та же архитектура
model2.load_state_dict(torch.load("best.pt", map_location="cpu", weights_only=True))
model2.eval()
```

Файл хранит **только тензоры и имена**. Класс модели нужен снова при загрузке.

In [78]:
# 2.1 сохранить state_dict и загрузить в новую модель
torch.manual_seed(0)
m = TinyReg()
# чуть «обучим» один шаг, чтобы веса ≠ init
opt = torch.optim.SGD(m.parameters(), lr=0.1)
xb, yb = next(iter(loader))
loss = nn.functional.mse_loss(m(xb), yb)
opt.zero_grad()
loss.backward()
opt.step()

path_sd = CKPT / "tiny_sd.pt"
torch.save(m.state_dict(), path_sd)

m2 = TinyReg()
m2.load_state_dict(torch.load(path_sd, map_location="cpu", weights_only=True))

# после load предсказания должны совпасть
with torch.no_grad():
    same = torch.allclose(m(xb), m2(xb))
print("preds equal:", same)

preds equal: True


In [79]:
need(path_sd.is_file(), "torch.save(m.state_dict(), path_sd)")
need(same is True, "load_state_dict — предсказания m и m2 должны совпасть")
# свежая модель без load ≠ m
m3 = TinyReg()
with torch.no_grad():
    need(not torch.allclose(m(xb), m3(xb)), "без load веса другие — ок")
print("2.1 — верно")

2.1 — верно


## 3. <span style="color:#7CDBA0">Лучшие веса по val</span> — главное для работы

**Главное в блоке:** после эпохи смотрим **val loss**; если лучше прежнего → `torch.save(state_dict)`. Так не теряем лучшие параметры.

**Не главное (чтобы не путаться):** какие тут `X`/`y`, формула `2x+1`, шум в ответах. Это бутафорские данные «лишь бы был цикл». Шум и формулу просто придумали; урок — про **save при улучшении val**, не про то, откуда взялись числа.

```python
best_val = float("inf")
if val_loss < best_val:
    best_val = val_loss
    torch.save(model.state_dict(), "best.pt")
```

**Сейчас в задании:** `val_loss < best_val`.

**Как делают на деле** (автор): не при любом микросдвиге, а при значимом — `best_val - val_loss > threshold` (абс.) или относительный порог в % (как у Plateau). Для учёбы порог можно отложить; в работе на длинных прогонах — полезен.

⚠️ `best_sd = model.state_dict()` **без** копии/`torch.save` — это ссылки: к концу обучения словарь испортится. На диск через `torch.save` — безопасно.

In [80]:
# 3.1 цикл: сохранять best по val_loss
# Суть: train → val → если val лучше → save. Данные X/y/шум — только подставка (см. markdown выше).
torch.manual_seed(1)
model = TinyReg()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
crit = nn.MSELoss()

# train 60 / val 20
n_tr = 60
Xtr, ytr = X[:n_tr], y[:n_tr]
Xva, yva = X[n_tr:], y[n_tr:]
tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True)

best_val = float("inf")
best_path = CKPT / "best_tiny.pt"
best_epoch = -1
hist_val = []

EPOCHS = 12
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in tr_loader:
        pred = model(xb)
        loss = crit(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_loss = crit(model(Xva), yva).item()
    hist_val.append(val_loss)

    # если val лучше — сохрани state_dict в best_path и обнови best_val / best_epoch
    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_path)
        print(f"  ↑ save epoch {epoch+1}  val={val_loss:.4f}")

print("best_epoch:", best_epoch, "best_val:", best_val)
print("hist_val:", [round(v, 4) for v in hist_val])

  ↑ save epoch 1  val=5.1016
  ↑ save epoch 2  val=3.8872
  ↑ save epoch 3  val=2.8698
  ↑ save epoch 4  val=1.9430
  ↑ save epoch 5  val=1.2203
  ↑ save epoch 6  val=0.6847
  ↑ save epoch 7  val=0.3190
  ↑ save epoch 8  val=0.1131
  ↑ save epoch 9  val=0.0226
  ↑ save epoch 10  val=0.0035
best_epoch: 9 best_val: 0.00349107151851058
hist_val: [5.1016, 3.8872, 2.8698, 1.943, 1.2203, 0.6847, 0.319, 0.1131, 0.0226, 0.0035, 0.0167, 0.035]


In [81]:
need(best_path.is_file(), "сохрани best state_dict в best_path")
need(best_epoch >= 0, "обнови best_epoch при улучшении")
need(abs(best_val - min(hist_val)) < 1e-9, "best_val должен быть min(hist_val)")
# загрузка best даёт val ≈ best_val (последняя эпоха может быть хуже)
restored = TinyReg()
restored.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True))
restored.eval()
with torch.no_grad():
    loaded_val = crit(restored(Xva), yva).item()
need(
    abs(loaded_val - best_val) < 1e-5, f"после load val={loaded_val}, ждали {best_val}"
)
print("3.1 — верно")

3.1 — верно


## 4. <span style="color:#7CDBA0">Checkpoint</span> — продолжить обучение

**Зачем:** блок 3 — «лучшая модель для predict». Здесь — «упал свет / Colab → не учить с нуля».

В файл кладут всё для resume: веса, состояние Adam, номер эпохи, история.

### В задании (упрощение)

Учимся эпохи 0…3 → **один** `torch.save` **после** цикла → «новая сессия» → с эпохи 4. Так проще понять load/resume.

### Как в бою

Свет может пропасть **внутри** цикла → checkpoint пишут **каждую эпоху** (или каждые N):

```python
best_val = float("inf")
for epoch in range(total):
    train_one_epoch(...)
    val_loss = validate(...)

    # лучшие веса для итогового predict
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best.pt")

    # последний checkpoint — чтобы продолжить после сбоя
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "epoch": epoch,
            "best_val": best_val,
            "hist_val": hist_val,
        },
        "last.pt",
    )

# после сбоя:
ckpt = torch.load("last.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model"])
opt.load_state_dict(ckpt["optimizer"])
for epoch in range(ckpt["epoch"] + 1, total):
    ...
```

Часто держат два файла: `best.pt` (качество) и `last.pt` (продолжить обучение).

In [84]:
# 4.1 сохранить checkpoint и продолжить с следующей эпохи
torch.manual_seed(2)
model = TinyReg()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
crit = nn.MSELoss()

hist_val = []
path_ckpt = CKPT / "resume.pt"
TOTAL = 8
SAVE_AT = 3  # «упали» после эпохи 3 (0-based: epoch==3 уже прошла)

for epoch in range(SAVE_AT + 1):
    model.train()
    for xb, yb in tr_loader:
        loss = crit(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        val_loss = crit(model(Xva), yva).item()
    hist_val.append(val_loss)

# сохрани словарь: model, optimizer, epoch(=SAVE_AT), hist_val
torch.save(
    {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "epoch": SAVE_AT,
        "hist_val": hist_val,
    },
    path_ckpt,
)

# --- «новая сессия» ---
model_r = TinyReg()
opt_r = torch.optim.Adam(model_r.parameters(), lr=0.05)
# torch.load
ckpt = torch.load(path_ckpt, map_location="cpu", weights_only=False)
# load_state_dict в model_r и opt_r
model_r.load_state_dict(ckpt["model"])
# следующая эпоха после сохранённой
start = ckpt["epoch"] + 1
model_r.load_state_dict(ckpt["model"])
opt_r.load_state_dict(ckpt["optimizer"])
hist_val = list(ckpt["hist_val"])

for epoch in range(start, TOTAL):
    model_r.train()
    for xb, yb in tr_loader:
        loss = crit(model_r(xb), yb)
        opt_r.zero_grad()
        loss.backward()
        opt_r.step()
    model_r.eval()
    with torch.no_grad():
        hist_val.append(crit(model_r(Xva), yva).item())

print("len hist:", len(hist_val), "start was", start)

len hist: 8 start was 4


In [85]:
need(path_ckpt.is_file(), "torch.save(ckpt, path_ckpt)")
need(start == SAVE_AT + 1, f"start должен быть {SAVE_AT + 1}, а {start}")
need(len(hist_val) == TOTAL, f"история должна иметь {TOTAL} эпох, а {len(hist_val)}")
print("4.1 — верно")

4.1 — верно


## 5. <span style="color:#7CDBA0">map_location</span>

Если сохраняли на GPU, а открываешь на CPU:

```python
torch.load(path, map_location="cpu", weights_only=True)
```

Без `map_location` будет ошибка «CUDA device not available». У нас всё на CPU — флаг всё равно привыкай писать.

In [86]:
# 5.1 load с map_location
sd = torch.load(best_path, map_location="cpu", weights_only=True)
need(isinstance(sd, dict), "должен вернуться state_dict")
print("keys:", list(sd.keys()))
print("5.1 — верно (просто запусти)")

keys: ['lin.weight', 'lin.bias']
5.1 — верно (просто запусти)


## Итог

| API | роль |
|:---|:---|
| `torch.save` / `torch.load` | файл ↔ объект |
| `model.state_dict()` / `load_state_dict` | снимок / вставка весов |
| `val_loss < best_val` → save | лучшая модель по валидации |
| checkpoint-словарь | продолжить после сбоя |
| `map_location` | GPU↔CPU |

Дальше: раздел **«Видео 13»** в `project/fashion_mnist.ipynb`. EarlyStopping + test — видео 14.